In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✅ SOLUTION
# 交叉熵损失 = -log(softmax(logits)[target])
# 直接算: -log(exp(logits[target]) / sum(exp(logits)))
# 但 exp(大 logits) 会溢出！所以用 log-sum-exp 恒等式:
#   log(softmax(x)) = x - log(sum(exp(x))) = x - logsumexp(x)

def cross_entropy_loss(logits, targets):
    # 第一步: 用 log-sum-exp 技巧计算 log-softmax（数值稳定）
    # logsumexp(logits, dim=-1) 计算 log(sum_j exp(z_j))，内部做了防溢出处理
    # keepdim=True 保持维度为 (B, 1)，方便和 (B, C) 的 logits 广播
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)

    # 第二步: 选出每个样本对应正确类别的 log 概率
    # torch.arange(B) 生成行索引 [0, 1, ..., B-1]
    # targets 是每行的列索引 → 得到 log_probs[i, targets[i]]
    # .mean() 对 batch 维度求平均
    return -log_probs[torch.arange(targets.shape[0]), targets].mean()

In [ ]:
# Demo
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets).item())
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets).item())

In [ ]:
from torch_judge import check
check('cross_entropy')